In [1]:
# cell 1 — imports and connection
import pandas as pd
import sqlite3
from pathlib import Path

DB_PATH = Path("../data/database/steam.db")
conn = sqlite3.connect(DB_PATH)

print("Connected to:", DB_PATH)
print("Tables available:")
pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

Connected to: ..\data\database\steam.db
Tables available:


,name
0,games
1,game_platforms
2,game_genres
3,game_tags


In [4]:
# cell 2 — genre performance ranking
query = """
SELECT 
    g.genre,
    COUNT(DISTINCT gm.appid)         AS total_games,
    ROUND(AVG(gm.owner_midpoint), 0) AS avg_owners,
    ROUND(AVG(gm.positive_ratio), 3) AS avg_positive_ratio,
    ROUND(AVG(gm.success_score), 2)  AS avg_success_score,
    ROUND(AVG(gm.price), 2)          AS avg_price
FROM game_genres g
JOIN games gm ON g.appid = gm.appid
WHERE gm.total_ratings > 10
AND g.genre NOT IN (
    'Early Access', 'Free to Play', 'Indie',
    'Gore', 'Violent', 'Nudity', 'Sexual Content',
    'Animation & Modeling', 'Design & Illustration',
    'Utilities', 'Audio Production', 'Video Production',
    'Web Publishing', 'Education', 'Software Training'
)
GROUP BY g.genre
HAVING total_games > 50
ORDER BY avg_success_score DESC
"""

genre_performance = pd.read_sql(query, conn)
print(genre_performance.to_string())

                   genre  total_games  avg_owners  avg_positive_ratio  avg_success_score  avg_price
0                    RPG         3417    201288.0               0.733              13.12       9.49
1                 Action         8811    264472.0               0.723              12.94       8.53
2              Adventure         7592    141864.0               0.732              12.87       8.38
3               Strategy         4053    186493.0               0.705              12.73       9.40
4  Massively Multiplayer          631    705753.0               0.625              12.66       5.19
5                 Casual         6835     74954.0               0.723              12.56       5.33
6                 Racing          760    109980.0               0.686              12.36      10.09
7                 Sports          919    108776.0               0.700              12.34      11.30
8             Simulation         4015    128041.0               0.682              12.27       9.76


In [6]:
# cell 3 — save genre performance query
with open("../sql/01_genre_performance.sql", "w") as f:
    f.write(query)

genre_performance.to_csv("../data/processed/genre_performance.csv", index=False)
print("Saved SQL and CSV")

Saved SQL and CSV


In [7]:
# cell 4 — price tier analysis
query2 = """
SELECT
    CASE
        WHEN price = 0                THEN '1. Free'
        WHEN price < 5                THEN '2. Under $5'
        WHEN price BETWEEN 5 AND 10   THEN '3. $5 - $10'
        WHEN price BETWEEN 10 AND 20  THEN '4. $10 - $20'
        WHEN price BETWEEN 20 AND 40  THEN '5. $20 - $40'
        ELSE                               '6. Over $40'
    END AS price_tier,
    COUNT(*)                             AS total_games,
    ROUND(AVG(owner_midpoint), 0)        AS avg_owners,
    ROUND(AVG(positive_ratio), 3)        AS avg_positive_ratio,
    ROUND(AVG(success_score), 2)         AS avg_success_score,
    ROUND(AVG(average_playtime), 1)      AS avg_playtime_mins
FROM games
WHERE total_ratings > 10
GROUP BY price_tier
ORDER BY price_tier
"""

price_tiers = pd.read_sql(query2, conn)
print(price_tiers.to_string())

     price_tier  total_games  avg_owners  avg_positive_ratio  avg_success_score  avg_playtime_mins
0       1. Free         2272    497027.0               0.718              13.13              521.8
1   2. Under $5         5386     55781.0               0.683              12.20               90.8
2   3. $5 - $10         7049    111023.0               0.738              12.78               85.8
3  4. $10 - $20         3954    191220.0               0.770              13.44              208.8
4  5. $20 - $40         1094    432221.0               0.764              13.92              506.9
5   6. Over $40          269    436190.0               0.719              13.95             1437.8


In [8]:
# cell 5 — save price tier
with open("../sql/02_price_tier_analysis.sql", "w") as f:
    f.write(query2)

price_tiers.to_csv("../data/processed/price_tiers.csv", index=False)
print("Saved")

Saved


In [9]:
# cell 6 — top 3 games per genre (window function)
query3 = """
WITH ranked_games AS (
    SELECT
        gm.name,
        gm.owner_midpoint,
        gm.positive_ratio,
        gm.success_score,
        gm.price,
        g.genre,
        RANK() OVER (
            PARTITION BY g.genre
            ORDER BY gm.success_score DESC
        ) AS genre_rank
    FROM games gm
    JOIN game_genres g ON gm.appid = g.appid
    WHERE gm.total_ratings > 50
    AND g.genre NOT IN (
        'Early Access', 'Free to Play', 'Indie',
        'Gore', 'Violent', 'Nudity', 'Sexual Content',
        'Animation & Modeling', 'Design & Illustration',
        'Utilities', 'Audio Production', 'Video Production',
        'Web Publishing', 'Education', 'Software Training'
    )
)
SELECT *
FROM ranked_games
WHERE genre_rank <= 3
ORDER BY genre, genre_rank
"""

top_games = pd.read_sql(query3, conn)
print(top_games.to_string())

                                                       name  owner_midpoint  positive_ratio  success_score  price                  genre  genre_rank
0                                                    Dota 2       150000000        0.858710      21.025230   0.00                 Action           1
1                                            Counter-Strike        15000000        0.973888      20.933580   9.13                 Action           2
2                                           Team Fortress 2        35000000        0.938107      20.780703   0.00                 Action           3
3                                                  Portal 2        15000000        0.986504      20.228553   9.13              Adventure           1
4                                                  Terraria         7500000        0.970398      20.207595   8.88              Adventure           2
5                                                  Unturned        35000000        0.902850      20.139761

In [11]:
# cell 7 — save top games per genre
with open("../sql/03_top_games_per_genre.sql", "w") as f:
    f.write(query3)

top_games.to_csv("../data/processed/top_games_per_genre.csv", index=False)
print("Saved")

Saved
